# Emailing Participants with API

This notebook is used to email participants with their API keys. This is not to be used by participants themselves, but rather by the workshop organizers to send out keys.

However, you can take a look at the code to see how bulk e-mailing is done using `sendgrid` 

Sendgrid (from Twilio) is a service that allows you to send emails in bulk. It is a paid service, but it has a free tier that allows you to send up to 100 emails per day.
You can use it to send emails to participants with their API keys.

You can sign up for a free account at [SendGrid](https://sendgrid.com/).

There are other services that allow you to send emails in bulk, such as [Mailgun](https://www.mailgun.com/) and [Amazon SES](https://aws.amazon.com/ses/), but we will use SendGrid for this workshop.

In old days this was done using `smtplib` and `email` libraries, however, these days most e-mail providers have limits on how many emails you can send per day, so it is better to use a service that is designed for this purpose.
This notebook will show you how to use SendGrid to send emails to participants with their API keys

In [ ]:
import os
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

# Store your key securely (e.g., using environment variable)
SENDGRID_API_KEY = os.getenv("SENDGRID_API")
if not SENDGRID_API_KEY:
    raise ValueError("SENDGRID_API environment variable not set")
else:
    print("SendGrid API key is set.") # again we do not want to print the key itself!! then anyone could use it....
sender_email = os.getenv("SENDGRID_FROM_EMAIL")
if not sender_email:
    raise ValueError("SENDGRID_FROM_EMAIL environment variable not set")
print("Sender email is set.")


## Reading e-mail participants from XLSX

Our participants are stored in an XLSX file, which we will read using `pandas`. We will then extract the email addresses and API keys from the DataFrame.



In [ ]:
# Load the provisioned participant spreadsheet from a private temp directory.
from pathlib import Path
import pandas as pd
print(f"pandas version: {pd.__version__}")

emails_file = Path("../temp/BSSDH_2025_provisioned_keys.xlsx")
if not emails_file.exists():
    raise FileNotFoundError(f"Emails file {emails_file} does not exist.")

df = pd.read_excel(emails_file, engine="openpyxl")
required_columns = {"E-mail", "Name", "Surname", "api_key"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required participant columns: {sorted(missing_columns)}")

print(f"Loaded participant table with {len(df)} rows and {len(df.columns)} columns.")
print("Expected participant columns are present.")


In [ ]:
# a slight complication is that E-mail column might actually contain multiple emails separated by commas or /
# our goal is to convert this dataframe into a list of dictionaries with following keys:
# 'emails' (list of emails (str)), 'name' (str), 'surname' (str), 'api_key' (str)
# so let's write a function that takes a row and returns a dictionary
import re
def row_to_dict(row):
    emails = emails = [e for e in re.split(r'[,\s/]+', row["E-mail"]) if e]
    return {
        'emails': emails,
        'name': row['Name'],
        'surname': row['Surname'],
        'api_key': row['api_key']
    }



In [ ]:
# now let's create that list of dictionaries
participants = df.apply(row_to_dict, axis=1).tolist()
print(f"Converted {len(participants)} participants to list of dictionaries.")

In [ ]:
# Count participants with more than one email address without printing personal data.
multiple_email_count = sum(1 for participant in participants if len(participant["emails"]) > 1)
print(f"Participants with multiple email addresses: {multiple_email_count}")


In [ ]:
import time

DELAY = 0.2  # delay in seconds to avoid hitting SendGrid rate limits
sent_count = 0
failed_count = 0

print(f"Sending emails with a delay of {DELAY} seconds to avoid rate limits.")
for participant_index, participant in enumerate(participants, start=1):
    for email_index, email in enumerate(participant["emails"], start=1):
        message = Mail(
            from_email=sender_email,
            to_emails=email,
            subject="Your Unique API Key for the BSSDH Workshop on August 7th",
            plain_text_content=(
                f"Dear {participant['name']},

"
                "Thank you for participating in our Using LLMs in Humanities Research via API workshop.

"
                f"Your unique API key is:
{participant['api_key']}

"
                "Please keep this key secure and do not share it with others.

"
                "This key is essential for actively participating in the workshop.
"
                "The official repository for this workshop is available at:
"
                "https://github.com/ValRCS/BSSDH_2025_workshop_LLM_API
"
                "We will provide instructions at the workshop on how to access the repository and use the provided materials.

"
                "See you on August 7th!

"
                "According to the workshop schedule on: https://www.digitalhumanities.lv/bssdh/2025/Programme/
"
                "The first session of this particular workshop will start at 11:30AM
"
                "There is another workshop before ours which does not require this key.

"
                "There is no need to reply to this e-mail as everything will be explained at the workshop.

"
                "Best regards,
"
                "On behalf of the BSSDH Workshop Team - Valdis Saulespurens
"
            )
        )
        try:
            sg = SendGridAPIClient(SENDGRID_API_KEY)
            response = sg.send(message)
            sent_count += 1
            print(f"Sent participant {participant_index}, address {email_index}. Status: {response.status_code}")
        except Exception as e:
            failed_count += 1
            print(f"Failed participant {participant_index}, address {email_index}: {type(e).__name__}")
        time.sleep(DELAY)

# SendGrid does not require explicit cleanup, but deleting the client can help VS Code notebooks stop cleanly.
if "sg" in locals():
    del sg
import gc
gc.collect()
print(f"All emails processed. Sent: {sent_count}; failed: {failed_count}.")
